In [ ]:
# Cell 1: Clone repo + install deps
!git clone https://github.com/drosadocastro-bit/cibuco-boriken
%cd cibuco-boriken
!pip install -q -r requirements.txt
print("Setup complete")

In [ ]:
# Cell 2: Kaggle credentials + data download
from google.colab import files
import os

uploaded = files.upload()  # upload kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

!kaggle competitions download -c birdclef-2026
!mkdir -p data/birdclef-2026
!unzip -q birdclef-2026.zip -d data/birdclef-2026
print("Data ready")

In [ ]:
# Cell 3: Mount Drive (save model after training)
from google.colab import drive
import os

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/cibuco_boriken/'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Drive mounted. Saving to: {SAVE_DIR}")

In [ ]:
# Cell 4: Training run (500 samples, 10 epochs)
!python -m birdclef.train \
  --backbone small \
  --epochs 10 \
  --max-samples 500 \
  --include-soundscapes

In [ ]:
# Cell 5: CFAR k-sweep on trained model
!python -m birdclef.evaluate_thresholds \
  --backbone small \
  --max-samples 500 \
  --include-soundscapes \
  --k-sweep 1.0 1.5 2.0 2.5 3.0

In [ ]:
# Cell 6: Display and save figure/model
from IPython.display import Image, display
import shutil

display(Image('k_sweep_figure.png'))

shutil.copy('k_sweep_figure.png', SAVE_DIR + 'k_sweep_500samples.png')
shutil.copy('birdclef/models/birdclef_model.pt', SAVE_DIR + 'birdclef_model_500samples.pt')
print("Saved to Drive")

In [ ]:
# Cell 7: Print paper-ready results table
import json
from pathlib import Path

results_path = Path('k_sweep_results.json')
if not results_path.exists():
    print('k_sweep_results.json not found. Run Cell 5 first.')
else:
    rows = json.loads(results_path.read_text(encoding='utf-8'))
    print('## Table 1: CFAR k-Sensitivity Results (500 samples)')
    print('| k | F1 Fixed | F1 CFAR | FPR Fixed | FPR CFAR | T_mean |')
    print('|---|----------|---------|-----------|----------|--------|')
    for r in rows:
        print(f"| {r['k']:.1f} | {r['f1_fixed']:.4f} | {r['f1_cfar']:.4f} | {r['fpr_fixed']:.4f} | {r['fpr_cfar']:.4f} | {r['threshold_mean']:.4f} |")